In [1]:
import pandas as pd
import json
# !pip install pandas
# !pip install openpyxl
from pandas.errors import EmptyDataError 
from datetime import datetime
import os
import numpy as np
from openpyxl import load_workbook


In [3]:
path = "D:/mac/zimttech/Projects/impilo-datim/datim_json_files/2025Q1/datim_files/json/"
file_list = []
# print(os.listdir(path))
def get_all_files(root_dir):
    all_files = []
    for root, directories, files in os.walk(root_dir):
        for file in files:
            all_files.append(os.path.join(root, file))
    all_files = [path.replace("\\","/").replace("._", "") for path in all_files]
    return all_files

for file in get_all_files(path):
    with open(f"{file}") as f:
        try:
            if f"{file}".endswith(".json"):
                json_file = f.read()
                json_file = json.loads(json_file)
                data_values = json_file["dataValues"]
                df_data = pd.DataFrame(data_values)
                df_data["period"] = json_file["period"]
                df_data["orgUnit"] = json_file["orgUnit"]
                file_list.append(df_data)
        except json.JSONDecodeError:
#             print(f"Invalid JSON format in {file}")
            print(file.split("/")[-1])
        except FileNotFoundError:
            print(f"File {path} not found")


df = pd.concat(file_list)
df["value"] = df["value"].replace('null', 0)
df["value"] = df["value"].astype("int")

df.drop_duplicates(inplace=True)
val_missing_df = df.copy()
df.head()

,dataElement,categoryOptionCombo,value,period,orgUnit
0,KP_PREV_N_TA_KeyPop_v2,mkXXjV42FM9,0,202410,ZW000A01
1,KP_PREV_N_TA_KeyPop_v2,EoZ7f4rkx2g,0,202410,ZW000A01
2,KP_PREV_N_TA_KeyPop_v2,hvgp9xnuUrx,0,202410,ZW000A01
3,KP_PREV_N_TA_KeyPop_v2,X5WOZxTBU2j,0,202410,ZW000A01
4,KP_PREV_N_TA_KeyPop_v2,etnBr84tFtJ,0,202410,ZW000A01


In [4]:
# Indicators to remove for reporting according to new PEPFAR MER Guildlines

unique_data_elements = [
    "HTS_RECENT_N_TA_Index_Age_Sex_RTRI_HIVStat",
    "HTS_RECENT_N_TA_Index_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_Inpat_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_Inpat_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_KPop_Confirm_HIVStatus",
    "HTS_RECENT_N_TA_KeyPop_RTRI_HIVStatus",
    "HTS_RECENT_N_TA_Other_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_Other_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_PMTCT_O_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_PMTCT_O_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_PMTCT_PB_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_PMTCT_PB_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_PMTCT_PP-LandD_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_PMTCT_PP-LandD_Confirm_A_S_HIV",
    "HTS_RECENT_N_TA_SNS_Age_Sex_RTRI_HIVStat",
    "HTS_RECENT_N_TA_SNS_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_STI_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_STI_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_TB_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_TB_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_VCT_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_VCT_Confirm_Age_Sex_HIV",
    "HTS_RECENT_N_TA_VMMC_Age_Sex_RTRI_HIV",
    "HTS_RECENT_N_TA_VMMC_Confirm_Age_Sex_HIV",
    "HTS_SELF_N_TA_KeyPop_HIVSelfTest",
    "HTC_TST_N_TA_KeyPop_Result",
    "KP_PREV_N_DSD_KeyPop_Status",
    "KP_PREV_N_DSD_KeyPop_v2",
    "KP_PREV_N_TA_KeyPop_Status",
    "KP_PREV_N_TA_KeyPop_v2",
    "PrEP_CT_N_DSD_KeyPop",
    "PrEP_CT_N_TA_KeyPop",
    "PrEP_NEW_N_TA_KeyPopAbr",
    "SC_CURR_N_NoApp_CurrentARVBottles",
    "TX_CURR_N_TA_KeyPop_HIVStatus",
    "TX_ML_N_TA_KeyPop_ARTNoContact_HIV",
    "TX_NEW_N_TA_KeyPop_HIVStatus",
    "TX_PVLS_D_TA_KeyPop_HIVStatus",
    "TX_PVLS_N_CS_Age_Sex_HIV",
    "TX_PVLS_N_CS_KeyPop_HIVStatus", 
    "TX_PVLS_N_TA_Age_Sex_HIV",
    "TX_PVLS_N_TA_KeyPop_HIVStatus",
    "TX_RTT_N_DSD_KeyPop_HIVStatus",
    "TX_RTT_N_TA_KeyPop_HIVStatus"
]

# Create a filter to identify rows to keep
filter_keep = ~df['dataElement'].isin(unique_data_elements)

# Apply the filter to get the filtered dataframe
df_filtered = df[filter_keep]
df_filtered.head(2)

,dataElement,categoryOptionCombo,value,period,orgUnit
5,PrEP_CT_N_TA_Age_Sex,BYmlmGMcCWx,0,202410,ZW000A01
6,PrEP_CT_N_TA_Age_Sex,zE5NFpGXDy4,0,202410,ZW000A01


In [5]:
def generate_datim_file(df_filtered, period=2025, quarter="Q1", path=""):
    """
    Generate DATIM file using filtered data, site details JSON, and new_temp JSON
    """
    # Set up parameters
    period_quarter = f'{period}{quarter}'
    quarters_dict = {
        "Q1": [f'{int(period) - 1}10', f"{int(period) - 1}11", f"{int(period) - 1}12"],
        "Q2": [f"{int(period)}01", f"{int(period)}02", f"{int(period)}03"],
        "Q3": [f"{int(period)}04", f"{int(period)}05", f"{int(period)}06"],
        "Q4": [f"{int(period)}07", f"{int(period)}08", f"{int(period)}09"]
    }
    
    q = quarters_dict.get(quarter)
    
    if len(df_filtered) > 0:
        # Filter by period
        df_filtered = df_filtered[df_filtered["period"].astype(str).isin(q)]
        print(f"Number of unique org units: {len(df_filtered['orgUnit'].unique())}")
        
        # Read the mapping data
        with open('./mapping/site_codes_orgunits_mapping.json', 'r') as json_file:
            site_codes_orgunits_mapping_dict = json.load(json_file)
        
        # Map site codes to datim org units
        df_filtered["orgUnit"] = df_filtered["orgUnit"].map(site_codes_orgunits_mapping_dict)
        
        # Load site details and new_temp from JSON
        with open('../mapping/site_details.json', 'r') as f:
            site_details_data = json.load(f)
        
        with open('./mapping/new_temp.json', 'r') as f:
            new_temp_data = json.load(f)
        
        site_details = pd.DataFrame(site_details_data)
        new_temp = pd.DataFrame(new_temp_data)
        
        # Define indicators aggregation type
        ehr_aggregated_ind = {
            "HTS_SELF": "quarterly",
            "PrEP_CT": "quarterly",
            "TB_STAT": "quarterly",
            "Tx_CURR": "quarterly",
            "Tx_RTT": "quarterly",
            "TX_PVLS": "yearly",
            "PMTCT_HEI_POS": "quarterly",
            "TX_ML": "quarterly",
            "TB_ART": "yearly",
            "TB_PREV": "bi-annual",
            "TX_TB": "bi-annual",
            "TX_NEW": "quarterly",
            "PMTCT_ART": "quarterly"
        }
        
        excludeKeywords = list(ehr_aggregated_ind.keys())
        excludeKeywords = [i.lower() for i in excludeKeywords]
        
        # Process indicators based on aggregation type
        ehr_aggregated_data = df_filtered[df_filtered["dataElement"].str.contains('(?i)' + '|'.join(excludeKeywords))]
        ehr_aggregated_data = ehr_aggregated_data[ehr_aggregated_data["period"] == q[-1]]
        
        # Filter out indicators that were not aggregated in EHR
        not_aggregated_data = df_filtered[~df_filtered["dataElement"].str.contains('(?i)' + '|'.join(excludeKeywords))]
        
        # Combine the data and aggregate
        complete_df = pd.concat([ehr_aggregated_data, not_aggregated_data])
        complete_df = complete_df.groupby(by=['dataElement', 'orgUnit', 'categoryOptionCombo'], as_index=False)['value'].sum()
        
        # Rename columns to match required datim template
        complete_df.rename(columns={"dataElement": "DATIM_Indicator", "orgUnit": "Org_Unit", "categoryOptionCombo": "Disaggregation"}, inplace=True)
        
        # Create the datim template structure
        org_id_data = pd.DataFrame()
        
        # Get valid org IDs (non-NaN)
        valid_org_ids = [org_id for org_id in set(complete_df["Org_Unit"]) if pd.notna(org_id)]

        # Create a list to store the number of rows in new_temp
        num_rows = len(new_temp)

        # Create a new column with a list of all valid org IDs repeated for each row in new_temp
        org_id_expanded = np.repeat(valid_org_ids, num_rows)

        # Create a dataframe where each row in new_temp is repeated for each org ID
        new_temp_expanded = pd.concat([new_temp] * len(valid_org_ids), ignore_index=True)

        # Add the org ID column
        new_temp_expanded["DATIM Org ID"] = org_id_expanded

        # Replace the original org_id_data with this expanded dataframe
        org_id_data = new_temp_expanded
        
        # Add standard columns
        org_id_data["ElementLocation"] = "Facility"
        
        # Create mapping dictionaries
        site_name_dict = dict(zip(site_details["orgunit"], site_details["site_name"]))
        district_dict = dict(zip(site_details["orgunit"], site_details["district"]))
        province_dict = dict(zip(site_details["orgunit"], site_details["province"]))
        
        # Map values to new columns
        org_id_data["SiteDistrict"] = org_id_data["DATIM Org ID"].map(district_dict)
        org_id_data["SiteName"] = org_id_data["DATIM Org ID"].map(site_name_dict)
        org_id_data["SiteProvince"] = org_id_data["DATIM Org ID"].map(province_dict)
        
        # Rename columns
        org_id_data.rename(columns={"DATIM Org ID": "Org_Unit", "DATIM Indicator": "DATIM_Indicator"}, inplace=True)
        
        # Add additional columns
        org_id_data["Funding_Mechanism"] = 70465
        org_id_data["Period"] = period_quarter
        org_id_data["ElementType"] = "TA"
        
        # Select columns for the template
        columns = [
            "Period", "DATIM_Indicator", "Org_Unit", "Disaggregation", "Funding_Mechanism", 
            "SiteProvince", "SiteDistrict", "SiteName", "ElementType", "ElementResultGroup",
            "ElementGender", "ElementAge", "ElementModality", "ElementLocation", "Indicator"
        ]
        
        # Get the base template
        datim_template = org_id_data[columns]
        
        # Merge with complete_df to get values
        datim_template = datim_template.merge(
            complete_df[["DATIM_Indicator", "Org_Unit", "Disaggregation", "value"]], 
            on=["DATIM_Indicator", "Org_Unit", "Disaggregation"], 
            how="left"
        )
        
        # Filter only rows with values
        datim_template = datim_template[datim_template["value"].notna()]
        
        # Rename and convert value column
        datim_template.rename(columns={"value": "Value"}, inplace=True)
        datim_template["Value"] = datim_template["Value"].astype('int')
        
        # Remove duplicates
        datim_template.drop_duplicates(keep="first", inplace=True)
        
        # Reorder columns as per requirement
        datim_template = datim_template[['Period', 'Funding_Mechanism', 'SiteProvince', 'SiteDistrict', 'SiteName', 'Org_Unit', 
                                      'Indicator', 'DATIM_Indicator', 'Disaggregation', 'ElementType', 'ElementResultGroup',
                                      'ElementAge', 'ElementGender', 'ElementLocation', 'ElementModality', 'Value']]
        
        # Save to Excel file with filename including current date
        date = datetime.today().strftime('%Y-%m-%d')
        
        # Handle Excel file size limitations
        max_rows = 1048576
        writer = pd.ExcelWriter(f"{path}DATIM-{date}.xlsx")
        current_sheet = 1
        current_length = 0
        
        for index, group in datim_template.groupby('DATIM_Indicator'):
            if current_length + len(group) > max_rows:
                # Start a new sheet if the current one is full
                current_sheet += 1
                current_length = 0
                # Write the group with headers since it's the start of a new sheet
                group.to_excel(writer, sheet_name=f'Sheet_{current_sheet}', startrow=current_length, header=True)
            else:
                # Write the group without headers for subsequent writes within the same sheet
                group.to_excel(writer, sheet_name=f'Sheet_{current_sheet}', startrow=current_length, header=current_length == 0)
            
            current_length += len(group)
        
        writer.close()
        
        print("Data saved to Excel file.")
        return datim_template
    else:
        print("No data to process after filtering.")
        return None


datim_file_df = generate_datim_file(df_filtered)
datim_file_df.head()


Number of unique org units: 621
Data saved to Excel file.


,Period,Funding_Mechanism,SiteProvince,SiteDistrict,SiteName,Org_Unit,Indicator,DATIM_Indicator,Disaggregation,ElementType,ElementResultGroup,ElementAge,ElementGender,ElementLocation,ElementModality,Value
37,2025Q1,70465,Manicaland,Makoni,Chinyudze - 100245 - Rural Health Centre,Txi298JjnJB,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,dh4TQ68p2SC,TA,Cervical Cancer - Negative,15-19,Female,Facility,First Time Screened,0
38,2025Q1,70465,Manicaland,Makoni,Chinyudze - 100245 - Rural Health Centre,Txi298JjnJB,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,pdCeAB4EYYM,TA,Cervical Cancer - Negative,20-24,Female,Facility,First Time Screened,0
39,2025Q1,70465,Manicaland,Makoni,Chinyudze - 100245 - Rural Health Centre,Txi298JjnJB,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,qgGxi9db8sQ,TA,Cervical Cancer - Negative,25-29,Female,Facility,First Time Screened,0
40,2025Q1,70465,Manicaland,Makoni,Chinyudze - 100245 - Rural Health Centre,Txi298JjnJB,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,ZLqwxAM0rDn,TA,Cervical Cancer - Negative,30-34,Female,Facility,First Time Screened,0
41,2025Q1,70465,Manicaland,Makoni,Chinyudze - 100245 - Rural Health Centre,Txi298JjnJB,CXCA_SCRN TA,CXCA_SCRN_N_TA_Age_Sex_HIV_Scrn_Visit,k6PpW7YsDek,TA,Cervical Cancer - Negative,35-39,Female,Facility,First Time Screened,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1623612,2025Q1,70465,Matabeleland South,Bulilima,Bezu - 100066 - Clinic,KYBy3FfZcPu,HTS_RECENT TA,HTS_RECENT_N_TA_PMTCT_PP_LandD_Age_Sex_RTRI_HIV,VA48CG3mpsO,TA,Recent RTRI,40-44,Female,Facility,Pregnant and L&D,0
1623613,2025Q1,70465,Matabeleland South,Bulilima,Bezu - 100066 - Clinic,KYBy3FfZcPu,HTS_RECENT TA,HTS_RECENT_N_TA_PMTCT_PP_LandD_Age_Sex_RTRI_HIV,bq7mvUzCqro,TA,Long-Term RTRI,45-49,Female,Facility,Pregnant and L&D,0
1623614,2025Q1,70465,Matabeleland South,Bulilima,Bezu - 100066 - Clinic,KYBy3FfZcPu,HTS_RECENT TA,HTS_RECENT_N_TA_PMTCT_PP_LandD_Age_Sex_RTRI_HIV,jinOHBR15Hx,TA,Recent RTRI,45-49,Female,Facility,Pregnant and L&D,0
1623615,2025Q1,70465,Matabeleland South,Bulilima,Bezu - 100066 - Clinic,KYBy3FfZcPu,HTS_RECENT TA,HTS_RECENT_N_TA_PMTCT_PP_LandD_Age_Sex_RTRI_HIV,CIEQirmxUtS,TA,Long-Term RTRI,50+,Female,Facility,Pregnant and L&D,0
